## OutOfMemoryError
Happens when JVM requires more memory but couldn't get it. Typically, it is due to lack of heap memory, but could also happen due to lack of metaspace space, thread stack, etc.

Below are some code samples that would result in `OutOfMemoryError` (when appropriate heap size is set):

In [ ]:
record Product (Integer id, String name) {}

List<Product> products = new ArrayList<>();

int i = 1;
while(true) {
    products.add(new Product(i, "Product#" + i));
    i++;
}

Reading large files into memory can also result in the error:

In [ ]:
byte[] data = Files.readAllBytes(Path.of("huge-5GB-file.log"));   // heap size set to 2GB

Since each platform thread requires memory, creating too many can also result in memory error:

In [ ]:
while (true) {
    new Thread(() -> {
        try { Thread.sleep(Long.MAX_VALUE); } catch (InterruptedException e) { }
    }).start();   // thread count grows without limit
}

## Heap Dump
Is a snapshot of heap memory of a Java application. We can obtain it in few different ways:
1. Configuring application to generate heap dump when it crashes due to `OutOfMemoryException`:  
   ```sh
   $ java -jar -XX:+HeapDumpOnOutOfMemoryError -XX:HeapDumpPath=heapdump.hcprof application.jar
   ```
   
2. We can take heap dump of a running application using `jcmd` utility:  
   ```sh
   # jcmd <PID> GC.heap_dump <path>
   $ jcmd 17297 GC.heap_dump ~/Desktop/gc.hprof
   ```
   
3. Through Spring Boot actuator available at `/actuator/heapdump`
   ```yaml
   management:
       endpoints:
           web:
               exposure:
                   include: heapdump,health,info
   ```
   It is very important that we secure this endpoint since the heap dump can contain private data.

## Analysis
The generated heap dump is a binary file which can be analysed using multiple different tool like VisualVM or IntelliJ.

**Summary Tab:** shows the summary of the heap dump:  

![Heap Dump Summary](./images/summary_heap_dump.jpg)

**Class List:** the left panel shows the list of classes with count of objects created per class:

![Heap Dump Objects List](./images/objects_list_heap_dump.jpg)

It shows all the classes at the time of heap dump. The columns mean:
- Shallow size: the memory taken by the object itself (its own fields only).
- Retained size: the memory that would be freed if this object were garbage collected.

**Biggest Objects Tab:** sorted list of single largest objects based on retained size:

![Biggest Objects](./images/biggest_objects_heap_dump.jpg)

The screenshot above shows the retained size of the first object is around 9MB which means there are ~9MB objects that are only reachable through this object.

**Object Details:** we can view the details per class and select any object of that class to view its details:

![Object Detail](./images/object_detail_heap_dump.jpg)

Shortest Paths lists only the referrers that lie on the shortest route from a GC root to this object. Incoming References on the other hand contains all the objects that directly point to the selected object:

![Object Detail](./images/incoming_references_heap_dump.jpg)